#### **List of all tables in silver layer:**

In [0]:
%sql
SHOW TABLES IN data_silver.silver; 

**THINGS TO OBSERVE FROM BELOW COMMAND**:

We cannot run a standard SQL UPDATE command directly on a Streaming Table. Streaming Tables in Databricks (and Spark Structured Streaming) are designed to be append-only from the perspective of the streaming engine. You can use Time Travel to see the current vs. previous state, however you won't notice any change in table since we have mostly done operations like FULL REFRESh which sets up DLT , refresh and setting table properties, This drops the table and recreates it from the source (Bronze), effectively updating all records. 

If you try to run UPDATE my_streaming_table SET col = 'x', the engine will block it to protect the integrity of the streaming checkpoints.


**Implications "Under the Hood"**
When you perform an operation that modifies a streaming table (like a Full Refresh or an APPLY CHANGES logic), several things happen:
- Checkpoint Reset: If you perform a Full Refresh, the Checkpoint (which tracks which Bronze files have been processed) is deleted. The pipeline starts reading from the very beginning of the Bronze table.
- Write-Ahead Log (WAL): Delta Lake records the change in the _delta_log. A streaming reader downstream will see these as new "appends" even if they were logically "updates" in an SCD Type 2 scenario.
- State Store: If your streaming table uses aggregations (e.g., window()), the "State Store" (RocksDB or In-Memory) tracks intermediate counts. An update/refresh clears this state, which can be computationally expensive for large datasets.
- Downstream Breakage: If you have a second streaming table reading from the first one, and you perform a "Full Refresh" on the first one, the downstream table will usually fail. This is because the "Source" has been recreated, and the downstream checkpoint no longer matches the source's new ID. You would need to refresh the entire "Medallion" chain.

In [0]:
%sql
DESCRIBE History data_silver.silver.fact_city_time_series

**Time Travel** based on different version of tables updated:

The reason you don't see data in the older versions after a Full Refresh in DLT is due to how DLT manages the lifecycle of Streaming Tables versus standard Delta tables.

When you trigger a Full Refresh on a Streaming Table, DLT performs a destructive operation to ensure data consistency.

In [0]:
#%sql
#select * from data_silver.silver.fact_city_time_series version as of 21;

Visiting back to the **current version** of the table updated which is 2 right now:

In [0]:
#%sql
#select * from data_silver.silver.fact_city_time_series;

**Restoring older back as main version again** 

**NOT RECOMMENDED TO RUN BELOW code for normal table objects** 

since we don't want to revert the changes done for cleaning the metrics column distributed across various dataset tablse created in silver layer and current version is 1 for the cleaned up data without any duplicates

In [0]:
#%sql
#RESTORE TABLE data_silver.silver.city_time_series TO VERSION AS OF older_version_number;



**Some more important points about the DELTA table operations done above:**

- **ACID Compliance**: The restore operation is fully ACID-compliant; if it fails mid-way, the table remains in its current state.

- **Data Retention**: You can only restore to versions that have not been removed by a VACUUM command. By default, Delta Lake retains 7 days of history.

- **Idempotency**: Because a restore is a versioned event, you can theoretically "time travel" back to the version before the restore if you made a mistake during the rollback as shown above for tables, not streaming tables or deleted/ dropped delta tables

**Unit Testing:**

The unit testing script provides a standardized method to validate that the Bronze-to-Silver ingestion process has executed correctly. It programmatically checks for data loss, record duplication, and environment readiness.

- **Table Existence**: Verifies that target Silver tables are present in the catalog before attempting analysis.

- **Row Count Integrity (Data Retention)**: Compares the source Bronze count against the target Silver count to ensure 100% data retention from the initial load through multiple runs.
 
- **Idempotency (No Duplicacy)**: Uses distinct count validation to prove that the Delta Merge logic successfully prevented duplicate records, even if the ingestion was executed multiple times.
 
- **Namespace Accuracy**: Utilizes 3-level namespace prefixes (Catalog.Schema.Table) imported from centralized configuration files.

In [0]:
import re
from pyspark.sql import functions as F

# --- 1. IMPORT CONFIGURATION ---
try:
    from config.schema_config import BRONZE_PREFIX, SILVER_PREFIX, SILVER_TABLES
    print("Successfully imported configuration.")
except ImportError:
    BRONZE_PREFIX = "data_bronze.bronze"
    SILVER_PREFIX = "data_silver.silver"
    SILVER_TABLES = [
        'dim_cities_crosswalk', 'dim_countycrosswalk_zillow', 'dim_datadictionary',
        'fact_city_time_series', 'fact_county_time_series', 'fact_metro_time_series',
        'fact_neighborhood_time_series', 'fact_state_time_series', 'fact_zip_time_series'
    ]

# --- 2. MODULAR TESTING FUNCTION ---
def run_silver_integrity_tests(dataset_list):
    """
    Tests integrity by mapping Silver (prefixed) to Bronze (base) tables.
    Example: dim_cities_crosswalk (Silver) -> cities_crosswalk (Bronze)
    """
    print(f"\n{'SILVER TABLE':<35} | {'BRONZE SOURCE':<25} | {'STATUS'}")
    print("-" * 90)
    
    test_results = []

    for silver_ds in dataset_list:
        try:
            # Step A: Derive Bronze name by stripping 'dim_' or 'fact_'
            # regex '^dim_|^fact_' matches these prefixes at the start of the string
            bronze_ds = re.sub(r'^(dim_|fact_)', '', silver_ds)
            
            bronze_tbl = f"{BRONZE_PREFIX}.{bronze_ds}"
            silver_tbl = f"{SILVER_PREFIX}.{silver_ds}"
            
            # Step B: Check existence
            if not spark.catalog.tableExists(silver_tbl):
                print(f"{silver_ds:<35} | {bronze_ds:<25} | FAIL (Silver Missing)")
                continue
            if not spark.catalog.tableExists(bronze_tbl):
                print(f"{silver_ds:<35} | {bronze_ds:<25} | FAIL (Bronze Missing)")
                continue

            # Step C: Row Count Validation
            b_count = spark.table(bronze_tbl).count()
            s_count = spark.table(silver_tbl).count()
            
            # Step D: Idempotency (Deduplication) check
            # We compare total rows vs distinct rows in the Silver table
            is_distinct = spark.table(silver_tbl).distinct().count() == s_count
            
            # Determine Status
            if b_count != s_count:
                status = f"WARN (B:{b_count} != S:{s_count})"
            elif not is_distinct:
                status = "FAIL (Duplicates Found)"
            else:
                status = "PASS"

            print(f"{silver_ds:<35} | {bronze_ds:<25} | {status}")
            test_results.append((silver_ds, bronze_ds, b_count, s_count, status))

        except Exception as e:
            print(f"{silver_ds:<35} | {bronze_ds if 'bronze_ds' in locals() else 'N/A':<25} | ERROR: {str(e)[:25]}")
    
    return test_results

if __name__ == "__main__":
    results = run_silver_integrity_tests(SILVER_TABLES)